In [1]:
"""
VAR Forecasting Model
=====================
Real-time recursive forecasts of log real TTF NG prices.
Specifications: VAR(1), VAR(AIC, p≤6).
Variables (K=4): log_real_ttf, hdd, log_storage_avg, log_lng_avg.
"""

import numpy as np
import pandas as pd
from statsmodels.tsa.api import VAR
import warnings
warnings.filterwarnings("ignore")

# ── Parameters ────────────────────────────────────────────────────────────────
HORIZONS    = [1, 3, 6, 9, 12, 15, 18, 21, 24]
EVAL_START  = "2015-01-01"
VAR_START   = "2012-02-01"
AIC_LAG_MAX = 6
INPUT_FILE  = "VAR_Input_Data.xlsx"
OUTPUT_FILE = "Output_VAR_forecasts.xlsx"

VARS = ["log_real_ttf", "hdd", "log_storage_avg", "log_lng_avg"]

SPECIFICATIONS = [
    ("VAR(1)",        1,    False),
    ("VAR(AIC,p<=6)", None, True),
]

# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_excel(INPUT_FILE, sheet_name="Sheet1")
df.columns = ["date", "real_ttf", "log_real_ttf", "hdd", "log_storage_avg", "log_lng_avg"]
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

df_var = df[df["date"] >= VAR_START].dropna(subset=VARS).reset_index(drop=True)

# ── Helper: actual real price for a given year-month ─────────────────────────
def get_actual(ym_str):
    m = df[df["date"].dt.to_period("M").astype(str) == ym_str]
    return m["real_ttf"].values[0] if len(m) == 1 else np.nan

# ── Helper: AIC lag selection ─────────────────────────────────────────────────
def select_aic_lag(data, max_lag):
    """Select lag order by AIC up to max_lag. Returns best p, minimum p=1."""
    best_aic, best_p = np.inf, 1
    for p in range(1, max_lag + 1):
        try:
            res = VAR(data).fit(p)
            if res.aic < best_aic:
                best_aic = res.aic
                best_p   = p
        except Exception:
            pass
    return best_p

# ── Main forecasting loop ─────────────────────────────────────────────────────
records = []
eval_origins = df_var[df_var["date"] >= EVAL_START]["date"].tolist()

print(f"VAR forecasting: {len(eval_origins)} origins x "
      f"{len(SPECIFICATIONS)} specs x {len(HORIZONS)} horizons")
print(f"Specifications:  {[s[0] for s in SPECIFICATIONS]}")
print(f"AIC cap:         {AIC_LAG_MAX} lags  (Baumeister et al. 2024)")
print()

for i, origin_date in enumerate(eval_origins):

    history = df_var[df_var["date"] <= origin_date][VARS].values
    T       = len(history)

    if i % 20 == 0:
        print(f"  Origin {i+1}/{len(eval_origins)}: "
              f"{origin_date.strftime('%Y-%m-%d')}  T={T}")

    for label, fixed_lag, use_aic in SPECIFICATIONS:

        p = select_aic_lag(history, AIC_LAG_MAX) if use_aic else fixed_lag

        if T <= p:
            continue

        try:
            res = VAR(history).fit(p)
            fc  = res.forecast(history[-p:], steps=max(HORIZONS))
        except Exception:
            continue

        for h in HORIZONS:
            records.append({
                "forecast_origin": origin_date.strftime("%Y-%m-%d"),
                "horizon":         h,
                "model":           label,
                "actual_month":    (origin_date + pd.DateOffset(months=h)).strftime("%Y-%m"),
                "forecast":        np.exp(fc[h - 1, 0]),
                "actual":          get_actual((origin_date + pd.DateOffset(months=h)).strftime("%Y-%m")),
                "lag_order_used":  p,
            })

# ── Save output ───────────────────────────────────────────────────────────────
results = pd.DataFrame(records)
results.to_excel(OUTPUT_FILE, index=False)

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print("=" * 60)
print("VAR FORECASTING COMPLETE")
print("=" * 60)
print(f"  Total rows:       {len(results)}")
print(f"  Forecast origins: {results['forecast_origin'].nunique()}")
print(f"  Output:           {OUTPUT_FILE}")
print()

for label, _, _ in SPECIFICATIONS:
    sub      = results[results["model"] == label]
    lag_dist = (sub.drop_duplicates("forecast_origin")["lag_order_used"]
                   .value_counts().sort_index().to_dict())
    print(f"  {label}: {sub['forecast_origin'].nunique()} origins  "
          f"lag dist: {lag_dist}")

print()
first_origin = results["forecast_origin"].min()
sample = results[results["forecast_origin"] == first_origin]
print(f"Sample — first origin ({first_origin}):")
print(sample[["model","horizon","lag_order_used",
              "actual_month","forecast","actual"]].to_string(index=False))

VAR forecasting: 132 origins x 2 specs x 9 horizons
Specifications:  ['VAR(1)', 'VAR(AIC,p<=6)']
AIC cap:         6 lags  (Baumeister et al. 2024)

  Origin 1/132: 2015-01-31  T=36
  Origin 21/132: 2016-09-30  T=56
  Origin 41/132: 2018-05-31  T=76
  Origin 61/132: 2020-01-31  T=96
  Origin 81/132: 2021-09-30  T=116
  Origin 101/132: 2023-05-31  T=136
  Origin 121/132: 2025-01-31  T=156

VAR FORECASTING COMPLETE
  Total rows:       2376
  Forecast origins: 132
  Output:           Output_VAR_forecasts.xlsx

  VAR(1): 132 origins  lag dist: {1: 132}
  VAR(AIC,p<=6): 132 origins  lag dist: {4: 61, 5: 19, 6: 52}

Sample — first origin (2015-01-31):
        model  horizon  lag_order_used actual_month  forecast    actual
       VAR(1)        1               1      2015-02 19.163333 22.938516
       VAR(1)        3               1      2015-04 17.991076 22.046423
       VAR(1)        6               1      2015-07 18.262482 20.679393
       VAR(1)        9               1      2015-10 19.8813